# dlgskill

> Read, search, and edit dialogs and notebooks through the llmsurgery `Dialog`/`Message` model

#| export
Use this whenever the question or edit concerns a notebook's content: messages, sources, outputs, prompts and their replies.

## The hierarchy

Notebook work happens at three levels, and picking the right level is most of using this module well:

- Content (this module): what the messages say and how they change. `summary_dlg`, `find_msgs`, `view_dlg`, the message editing operations, and `reply2dlg`/`dlg2reply` for a prompt's reply.
- Representation (`fastcore.nbio`, formerly `execnb.nbio`): which keys exist, whether a file is schema-valid, whether bytes changed. Start with `validate_nb`/`validate_cell`, which name the offending cell; use `read_nb` directly when the question is about the dict itself.
- Raw text: only when the file will not parse at all.

Dropping a level is correct exactly when the question is about the representation rather than the content ("why does Jupyter reject this file?", "did that write change any bytes?"). Treat each drop as a signal, though: needing nbio or raw JSON to answer a content question means a higher-level tool was missing. Re-read this module's docs to check it truly is missing, and then propose adding it rather than repeating the workaround.

#| export
## Core APIs

Every function takes `dlg` as a `Dialog`, an ipynb path, or None meaning the current dialog file (`set_dlg`/`cur_dlg`); changes persist automatically only for file-backed dialogs.

- `summary_dlg(dlg)`: one preview line per message.
- `find_msgs(pattern, dlg, ...)`: search by regex, type, error state, heading, ids, or a `pred` (`symdef_finder`/`symref_finder`/`ast_finder` build structural ones); `context` defaults to 1 (the neighbouring message usually explains the match). Returns live `Message` objects in a `Msgs` list (`.show(maxlen)` for display control), so results are edited directly rather than re-addressed. Every message carries a `dlg` backref to its owning `Dialog`, so dialog-level operations are always in reach from a message in hand (e.g. `m.dlg.save()` after mutating `m.output` directly).
- `view_dlg(dlg_or_msgs)` / `view_msg(id)` / `view_msgs(*ids)` / `msg2xml(m)`: full views in the shared `item2xml` grammar (a prompt's reply is its `<out>` section); `view_dlg` also renders a `find_msgs` result.
- Structure: `add_msg`, `del_msgs`, `move_msgs`, `split_msg`, `merge_msgs`, `copy_msgs`/`cut_msgs`/`paste_msgs`, `create_dlg`; the `%%add_msg` magic takes verbatim bodies.
- Text edits: `msg_str_replace`, `msg_strs_replace`, `msg_insert_line`, `msg_replace_lines`, `msg_del_lines` (all with `re_filter`/line-range powers; `out=True` edits a prompt's reply or a code message's outputs literal); `python_msgs`/`ast_msgs` for structural rewrites; `lnhashview_msg`/`exhash_msg` for hash-verified line edits (`lnhashview_msg` is `view_msg(..., lnhashs=True)`; only `exhash_msg` needs the `exhash` package).
- `reply2dlg(pmsg)`/`dlg2reply(sub)`: explode a reply into note/code messages and back; byte-exact for fmt2hist-clean replies.

#| export
## Idiomatic usage

Start by registering the notebook: `set_dlg(path)` makes every function here default to it (and clikernel's `%nbrun` magic follows suit), so calls read as "do this to message X". Then orient before acting:

- `summary_dlg()` first for anything sizable: a cheap one-line-per-message map. `view_dlg()` when you need the full story in order with ids. Read a notebook in full before describing or changing what it does - the interleaved prose, examples, and stored outputs (`incl_out=True`) are the design rationale.
- Read before probing: an ad-hoc "what happens if..." experiment usually re-derives, more slowly and less reliably, what an existing example cell already shows. If after reading you still need an experiment, that's a gap in the notebook - add it as a proper example cell so the next reader doesn't repeat it.
- `find_msgs` is the targeted view: keep the default `context=1` (the neighbouring markdown usually explains the match), and `ids=` with context is the positional query ("does A precede B?", "what's the idiom around here?"). Name questions are structural, not textual: `symdef_finder`/`symref_finder`/`ast_finder` beat regexes over binding syntax. Where available, rgapi's `nbrg` searches cell sources across files and returns the cell ids these functions take.
- Edit at the right level: `add_msg`/`%%add_msg` for new messages; within a message, prefer hash-verified addressing (`lnhashview_msg`/`exhash_msg`, or exhash's `lnhashview_cell`/`exhash_cell` by path and cell id) - view with the lnhash variant the moment an edit is plausible, so the view doubles as the edit's address book. The plain `msg_*` editors fit where exhash can't express the edit, e.g. one `msg_str_replace` across an id list. Never splice via `read_nb`/`write_nb` internals: if a primitive is missing here, stop and propose adding it.
- Every editing operation returns a diff: read it - that's the immediate verification. `dlg.validate()` catches model-level damage early; `dlg.save()` writes back.
- Don't hand-roll `json`/`grep`/`bash` over ipynb files for anything these functions cover, and don't wrap calls in defensive scaffolding (`hasattr`, `try/except`) - call directly and read the bare result.

In [ ]:
#| default_exp dlgskill

In [ ]:
#| export
import shlex, re, copy
from fastcore.utils import *
from fastcore.meta import splice_sig
from fastcore.xtras import str_diff
from fastcore.xml import to_xml
from fastcore.nbio import item2xml
from fastcore.tools import insert_line, str_replace, strs_replace, replace_lines, del_lines, ast_replace, lnhash
from llmsurgery.dialog import *
from llmsurgery.ipynb import read_ipynb, write_ipynb
from llmsurgery.hist import reply2dlg, dlg2reply, render_outputs_ai  # chkstyle: ignore (re-exported via _all_)
_all_ = ['reply2dlg', 'dlg2reply']

In [ ]:
from fastcore.test import *
from tempfile import mkdtemp

## The current dialog, and finding messages

In [ ]:
#| export
_cur_dlg = None

def set_dlg(
    fname, # ipynb path that later calls will default to
):
    "Set the current dialog file, used by these functions when `dlg` is None"
    global _cur_dlg
    _cur_dlg = Path(fname)
    return _cur_dlg

def cur_dlg(): return _cur_dlg

def _to_dlg(x):
    "`x` as a `Dialog`: pass through, or read the ipynb at that path (None: the current dialog file)"
    if x is None: x = _cur_dlg
    if x is None: raise ValueError('No dialog: pass one, or call set_dlg() first')
    return x if isinstance(x, Dialog) else read_ipynb(x)

def _load_dlg(x):
    "`(dialog, we_loaded)`: like `_to_dlg`, plus whether this call read it from disk (and so should save it)"
    return _to_dlg(x), not isinstance(x, Dialog)

def _save(dlg, we_loaded):
    "Persist only when this call loaded `dlg` from a path; a live `Dialog` argument is the caller's to save"
    if we_loaded: dlg.save()

In [ ]:
#| export
@patch
def msg(self:Dialog,
    id, # A `Message` (matched by its id), or an id: exact, or unique prefix
):
    "The matching `Message` in this dialog"
    if isinstance(id, Message): id = id.id
    ms = [m for m in self.messages if m.id==id]
    if not ms: ms = [m for m in self.messages if m.id.startswith(id)]
    if len(ms) != 1: raise KeyError(f"{'ambiguous' if ms else 'no'} message id: {id!r}")
    return ms[0]

On-disk cell ids are message ids without the leading underscore, so `msg` accepts either form, and unique prefixes work like everywhere else in this ecosystem. The fixture below is written to disk too, since path-based calls re-read the file per call:

In [ ]:
tdir = Path(mkdtemp())
d = Dialog('demo')
setup = d.mk_message('# Setup', msg_type=snote)
calc = d.mk_message('x = 21\nx*2', msg_type=scode, output=code_output('42'))
ask = d.mk_message('Double it.', msg_type=sprompt, output='It is **42**.')
bad = d.mk_message('1/0', msg_type=scode, output=[dict(output_type='error', ename='E', evalue='boom', traceback=['tb'])])
write_ipynb(d, tdir/'demo.ipynb')
test_eq(d.msg(calc.id), calc)
test_eq(d.msg(calc.id[:5]), calc)  # unique prefixes work too
with expect_fail(KeyError, 'no message'): d.msg('zzzz')

## Views

In [ ]:
#| export
def summary_dlg(
    dlg=None, # A `Dialog`, ipynb path, or iterable of `Message`s (e.g. a `find_msgs` result); current dialog file if None
    maxlen:int=120, # Maximum characters per line
):
    "One `preview` line per message"
    ms = Msgs(dlg) if isinstance(dlg, (list, L)) else _to_dlg(dlg).messages
    return ms.show(maxlen)

In [ ]:
s = str(summary_dlg(d))
test_eq(len(s.splitlines()), 4)
assert 'reply(13)' in s and '# Setup' in s

In [ ]:
#| export
_t2tag = dict(note='markdown')

def msg2xml(m, incl_out=False, trunc_out=True, ids=True):
    "One message as concise XML: content bare inside its type tag, with an `<out>` section for a code output or a prompt's reply"
    if m.msg_type==sprompt: o = m.ai_res
    elif incl_out and m.msg_type==scode and m.output:
        o = render_outputs_ai(m.output)
        if trunc_out: o = truncstr(o, 512)
    else: o = ''
    it = item2xml(_t2tag.get(m.msg_type, m.msg_type), m.content, o, id=m.id if ids else None,
                  kind=m.meta.get('rec_kind'), meta=m.meta)
    return to_xml(it, do_escape=False)

def view_dlg(
    dlg=None, # A `Dialog`, ipynb path, or iterable of `Message`s (e.g. a `find_msgs` result); current dialog file if None
    incl_out:bool=False, # Include code outputs?
    only_errors:bool=False, # Show only code messages with error outputs (implies `incl_out`)?
    trunc_out:bool=True, # Truncate each output to ~512 chars?
):
    "Dialog (or a selection of its messages) as concise XML"
    if isinstance(dlg, (list, L)): nm, ms = 'msgs', list(dlg)
    else:
        d = _to_dlg(dlg)
        nm, ms = d.name, d.messages
    if only_errors: ms = [m for m in ms if m.has_error]
    body = ''.join(msg2xml(m, incl_out or only_errors, trunc_out) for m in ms)
    return PrettyString(f'<dialog name="{nm}">{body}</dialog>')

In [ ]:
v = str(view_dlg(d, incl_out=True))
assert '<prompt id="'+ask.id+'">Double it.<out>It is **42**.</out></prompt>' in v  # a prompt's reply is its out
assert '>x = 21\nx*2<out>42</out></code>' in v
verr = str(view_dlg(d, only_errors=True))
assert '1/0' in verr and 'Setup' not in verr

In [ ]:
#| export
def view_msg(
    m, # A `Message`, or an id looked up in `dlg`
    dlg=None, # `Dialog` or path when `m` is an id; the current dialog file if None
    nums:bool=True, # Show line numbers?
    start_line:int=1, # Starting line to view
    end_line:int=None, # End line (defaults to last line if None; -1 for EOF)
    lnhashs:bool=False, # Show exhash `lineno|hash|` addresses instead of line numbers?
):
    "Show message content with optional line numbers or exhash addresses"
    if not isinstance(m, Message): m = _to_dlg(dlg).msg(m)
    lines = m.content.splitlines()
    lines = lines[start_line-1:len(lines) if end_line in (None,-1) else end_line]
    return PrettyString('\n'.join((lnhash(i,l)+l if lnhashs else f'{i}: {l}' if nums else l) for i,l in enumerate(lines, start_line)))

def view_msgs(
    *ms, # `Message`s or ids
    dlg=None, # `Dialog` or path for id lookup; the current dialog file if None
    nums:bool=True, # Show line numbers?
    lnhashs:bool=False, # Show exhash `lineno|hash|` addresses instead of line numbers?
):
    "Show several messages, each preceded by a `# msg <id>` header"
    return PrettyString('\n'.join(f"# msg {(m if isinstance(m, Message) else _to_dlg(dlg).msg(m)).id}\n{view_msg(m, dlg, nums, lnhashs=lnhashs)}" for m in ms))

In [ ]:
test_eq(str(view_msg(calc)), '1: x = 21\n2: x*2')
test_eq(str(view_msg(calc, start_line=2)), '2: x*2')
test_eq(str(view_msg(calc, lnhashs=True)), f'{lnhash(1, "x = 21")}x = 21\n{lnhash(2, "x*2")}x*2')
assert str(view_msgs(setup, calc, nums=False)).startswith(f"# msg {setup.id}")

## Searching

In [ ]:
#| export
def _match_head(m, want):
    "Does `m` head the section `want` names? Exact first line when `want` starts with '#', hash-stripped text otherwise"
    if not m.header_level(): return False
    fl = m.content.split('\n', 1)[0]
    if want.startswith('#'): return fl.rstrip()==want.rstrip()
    return fl.lstrip('#').strip()==want.strip()

def find_msgs(
    re_pattern:str='', # Regex over content (a prompt's reply included), DOTALL+MULTILINE; an invalid regex matches literally
    dlg=None, # A `Dialog`, or ipynb path; the current dialog file if None
    msg_type:str=None, # Optional limit by type ('code', 'note', 'prompt', or 'raw')
    only_err:bool=False, # Only code messages with error outputs?
    only_exp:bool=False, # Only exported messages (nbdev export directive in content or meta)?
    ids='', # Optionally filter by ids (comma-separated str, or list); results are always in dialog order, whatever order the ids are given
    before:int=0, # Also include n messages before each match
    after:int=0, # Also include n messages after each match
    context:int=None, # Messages of context around matches (default 1, or 0 when `headers_only`)
    limit:int=None, # Max matched messages
    use_case:bool=False, # Case-sensitive matching?
    use_regex:bool=True, # Regex matching (else plain substring)?
    headers_only:bool=False, # Only heading notes (an outline view)?
    header_section:str=None, # Return the section starting with this heading, plus its children
    pred:callable=None, # Extra match criterion, e.g. from `symdef_finder`/`symref_finder`/`ast_finder`, or host-specific flags
)->Msgs: # Live messages, so results can be edited directly
    "Find messages in `dlg` (a `Dialog`, path, or iterable of messages) matching all the given criteria"
    if isinstance(dlg, (list, L)): d, ms = None, Msgs(dlg)
    else:
        d = _to_dlg(dlg)
        ms = d.messages
    if context is None: context = 0 if headers_only else 1
    if header_section is not None:
        head = first(m for m in ms if _match_head(m, header_section))
        ms = section_msgs(ms, head) if head else Msgs()
    if context: before = after = context
    flags = re.DOTALL|re.MULTILINE|(0 if use_case else re.IGNORECASE)
    if use_regex and re_pattern:
        try: re.compile(re_pattern)
        except re.error: use_regex = False
    pat = re.compile(re_pattern if use_regex else re.escape(re_pattern), flags) if re_pattern else None
    if isinstance(ids, str): ids = [o for o in ids.split(',') if o.strip()]
    if ids: idset = {d.msg(i.strip()).id for i in ids} if d else {i.strip() for i in ids}
    else: idset = None
    def _txt(m): return m.content + ('\n'+m.ai_res if m.msg_type==sprompt and m.ai_res else '')
    def _ok(m):
        if headers_only and not m.header_level(): return False
        if msg_type and m.msg_type!=msg_type: return False
        if only_err and not m.has_error: return False
        if only_exp and not m.exported: return False
        if pred and not pred(m): return False
        if idset is not None and m.id not in idset: return False
        return not pat or bool(pat.search(_txt(m)))
    hits = [i for i,m in enumerate(ms) if _ok(m)]
    if limit is not None: hits = hits[:limit]
    if before or after:
        keep = set()
        for i in hits: keep.update(range(max(0,i-before), min(len(ms), i+after+1)))
        hits = sorted(keep)
    return Msgs([ms[i] for i in hits])

`find_msgs` returns the messages themselves, so a search result can be edited in place. A prompt matches on its request or its reply; an invalid regex matches literally (search text arrives verbatim from humans); `pred=` takes any extra criterion, the channel hosts pass their own flag logic through; `header_section` follows note-message headings; `headers_only` gives an outline. `context` defaults to 1, since the neighbouring message is usually the explanation of whatever matched; pass `context=0` for exact hits only:

In [ ]:
test_eq(find_msgs('42', d, context=0), [ask])  # matches the reply text
test_eq(find_msgs(r'x\*2', d, msg_type=scode, context=0), [calc])
test_eq(find_msgs(dlg=d, only_err=True, context=0), [bad])
test_eq(len(find_msgs(r'x\*2', d)), 3)  # context defaults to 1: the match plus both neighbours
test_eq(find_msgs(dlg=d, headers_only=True), [setup])
test_eq(find_msgs(header_section='Setup', dlg=d, context=0), list(d.messages))  # everything under the heading
test_eq(find_msgs('42', d.messages[:3], context=0), [ask])  # any message iterable works too
find_msgs('42', d, context=0)[0].content = 'Double it, please.'
test_eq(ask.content, 'Double it, please.')  # live objects: the edit landed on the dialog
test_eq(find_msgs('*42*', d, context=0), [ask])  # an invalid regex falls back to a literal match
test_eq(find_msgs(dlg=d, pred=lambda m: m.msg_type==sprompt, context=0), [ask])  # host-specific extra criterion
xd = Dialog('exports')
exp1 = xd.mk_message('#| export\ndef f(): pass', msg_type=scode)
exp2 = xd.mk_message('def g(): pass', msg_type=scode)
exp2.exported = True
deep = xd.mk_message('## Deep\nbody under the heading', msg_type=snote)
test_eq(find_msgs(dlg=xd, only_exp=True, context=0), [exp1, exp2])
test_eq(find_msgs(header_section='Deep', dlg=xd, context=0), [deep])  # matched on the first line only
test_eq(find_msgs(header_section='## Deep', dlg=xd, context=0), [deep])  # a '#'-prefixed argument pins the level
test_eq(find_msgs(header_section='# Deep', dlg=xd, context=0), [])

## Structural operations

In [ ]:
#| export
def move_msgs(
    ids, # Message(s) or id(s) to move
    before=None, # Move before this message or id
    after=None, # Move after this message or id
    dlg=None, # A `Dialog`, or ipynb path; the current dialog file if None
):
    "Move messages, keeping their relative order; returns them"
    d, sv = _load_dlg(dlg)
    if (before is None) == (after is None): raise ValueError('Exactly one of before/after required')
    ms = Msgs([d.msg(i) for i in listify(ids)])
    d.remove_msgs(ms)
    idx = d.messages.index(d.msg(before if before is not None else after)) + (after is not None)
    d.messages[idx:idx] = ms
    _save(d, sv)
    return ms

In [ ]:
#| export
def _atts_for(content, att_map):
    "The attachments from `att_map` whose ids are referenced in `content`"
    return [a for aid,a in att_map.items() if aid in content]

def split_msg(
    id, # Message or id to split
    *linenos:int, # Split before each of these 1-based lines
    dlg=None, # A `Dialog`, or ipynb path; the current dialog file if None
):
    "Split a message into pieces (returned as `Msgs`; the first keeps its id): one `'\\n\\n'` is absorbed at each cut (so `merge_msgs` restores blank-line-separated content byte-exactly), `meta_attrs` fields, the cell `meta`, and a leading `#| export` copy to every piece, attachments follow their references, and unreferenced ones stay on the first piece"
    d, sv = _load_dlg(dlg)
    m = d.msg(id)
    lines = m.content.splitlines()
    cuts = [0, *[l-1 for l in linenos], len(lines)]
    parts = ['\n'.join(lines[a:b]) for a,b in zip(cuts, cuts[1:])]
    for i in range(len(parts)-1):  # each cut absorbs the '\n\n' that a later merge re-inserts
        if   parts[i].endswith('\n'):     parts[i] = parts[i][:-1]
        elif parts[i+1].startswith('\n'): parts[i+1] = parts[i+1][1:]
    exp = re.match(r'#\|\s*exports?[^\n]*\n', m.content)
    keep = {a: getattr(m, a) for a in m.meta_attrs if hasattr(m, a)}
    att_map = {a.id: a for a in m.attachments}
    refs = [_atts_for(p, att_map) for p in parts]
    used = {a.id for r in refs for a in r}
    refs[0] += [a for a in m.attachments if a.id not in used]
    m.content, m.attachments = parts[0], refs[0]
    prev, res = m, [m]
    for p,r in zip(parts[1:], refs[1:]):
        if exp and not p.startswith('#|'): p = exp.group(0) + p
        prev = d.mk_message(p, after=prev, msg_type=m.msg_type, attachments=r, meta=copy.deepcopy(m.meta), **keep)
        res.append(prev)
    _save(d, sv)
    return Msgs(res)


def merge_msgs(
    *ids, # Adjacent messages or ids to merge
    dlg=None, # A `Dialog`, or ipynb path; the current dialog file if None
):
    "Merge into the first message (returned, keeping its id): same-type merges keep the type, mixed become a note via `merge_content`; metas and directives merge first-wins via `merge_parts`, outputs clear, attachments combine"
    d, sv = _load_dlg(dlg)
    ms = [d.msg(i) for i in ids]
    mtype = ms[0].msg_type if len(set(m.msg_type for m in ms))==1 else snote
    content, meta = merge_parts(ms, [m.merge_content(mtype) for m in ms])
    ms[0].update(content=content, meta=meta, output=None, msg_type=mtype, attachments=L(ms).attrgot('attachments').concat())
    d.remove_msgs(ms[1:])
    _save(d, sv)
    return ms[0]

In [ ]:
sd = Dialog('s')
sm = sd.mk_message('#| export\ndef f(): pass\ndef g(): pass', msg_type=scode, meta={'k': 1})
i1,i2 = split_msg(sm, 3, dlg=sd)
test_eq([m.content for m in sd.messages], ['#| export\ndef f(): pass', '#| export\ndef g(): pass'])
test_eq([m.meta for m in sd.messages], [{'k': 1}, {'k': 1}])  # whole cell meta copies to every piece
i2.meta['k'] = 2  # pieces don't share the dict
test_eq(i1.meta, {'k': 1})
merge_msgs(i1, i2, dlg=sd)
test_eq((sd.messages[0].content, sd.messages[0].meta), ('#| export\ndef f(): pass\n\ndef g(): pass', {'k': 1}))
hd = Dialog('hoist')
h1, h2 = hd.mk_message('def a(): pass'), hd.mk_message('#| export\ndef b(): pass')
hm = merge_msgs(h1, h2, dlg=hd)  # any exported piece exports the merge, via meta
test_eq((hm.content, hm.meta, hm.exported), ('def a(): pass\n\ndef b(): pass', {'nbdev': {'export': 'true'}}, True))
sd.mk_message('tail', msg_type=snote)
move_msgs(i1, after=sd.messages[-1], dlg=sd)
test_eq([m.content[:4] for m in sd.messages], ['tail', '#| e'])

In [ ]:
ad = Dialog('atts')
at1, at2 = Attachment(b'x', 'text/plain'), Attachment(b'y', 'text/plain')
am = ad.mk_message(f'see attachment:{at1.id}\nplain tail', msg_type=snote, attachments=[at1, at2])
p1, p2 = split_msg(am, 2, dlg=ad)
test_eq(len(p1.attachments), 2)  # the referenced one, plus the unreferenced one kept on the first piece
test_eq(p2.attachments, [])
jd = Dialog('junction')
j = jd.mk_message('a\n\n\nb', msg_type=snote)
j1, j2 = split_msg(j, 4, dlg=jd)
test_eq((j1.content, j2.content), ('a\n', 'b'))  # one '\n\n' absorbed at the cut: exactly what merge re-inserts
test_eq(merge_msgs(j1, j2, dlg=jd).content, 'a\n\n\nb')  # ...so blank-line-separated content round trips byte-exactly
k1, k2 = split_msg(jd.mk_message('a\n\nb', msg_type=snote), 3, dlg=jd)
test_eq((k1.content, k2.content), ('a', 'b'))
w1, w2 = split_msg(jd.mk_message('a\n\n\nb', msg_type=snote), 3, dlg=jd)  # cut on the blank line: residue stays on the right
test_eq((w1.content, w2.content), ('a', '\nb'))
with expect_fail(ValueError, 'Exactly one'): move_msgs(w2, dlg=jd)

In [ ]:
#| export
_paste_buf = []

def copy_msgs(
    *ids, # Messages or ids to copy
    dlg=None, # A `Dialog`, or ipynb path; the current dialog file if None
):
    "Copy messages into the paste buffer (replacing its contents), for later `paste_msgs`"
    global _paste_buf
    d = _to_dlg(dlg)
    _paste_buf = Msgs([d.msg(i) for i in ids])
    return _paste_buf

def cut_msgs(
    *ids, # Messages or ids to cut
    dlg=None, # A `Dialog`, or ipynb path; the current dialog file if None
):
    "Copy messages into the paste buffer, then remove them from the dialog"
    d, sv = _load_dlg(dlg)
    res = copy_msgs(*ids, dlg=d)
    d.remove_msgs(res)
    _save(d, sv)
    return res

def paste_msgs(
    before=None, # Insert before this message or id
    after=None, # Insert after this message or id
    dlg=None, # A `Dialog`, or ipynb path; the current dialog file if None
):
    "Insert copies of the buffered messages (fresh ids) before/after a message or id; returns the new messages"
    d, sv = _load_dlg(dlg)
    if before is not None: before = d.msg(before)
    if after is not None: after = d.msg(after)
    res = Msgs(d.mk_messages(_paste_buf, before=before, after=after))
    _save(d, sv)
    return res

In [ ]:
copy_msgs(i1, dlg=sd)
pasted = paste_msgs(after=i1, dlg=sd)
test_eq(len(sd.messages), 3)
assert pasted[0].id != sd.msg(i1).id and pasted[0].content == sd.msg(i1).content
cut_msgs(pasted[0].id, dlg=sd)
test_eq(len(sd.messages), 2)

## Editing message text

The editing operations run fastcore.tools' text primitives over a message's content, or, with `out=True`, over a prompt's reply markdown (assignment re-wraps it through `prompt_output`). Only prompts have text-editable output: for a code message's outputs, assign `msg.output` directly. The old behavior of editing a prompt cell's joined request+reply source is retired. In bulk mode (a list of ids, or 'all'), no-ops are dropped from the result but errors are reported.

In [ ]:
#| export
def _out_text(m):
    "A prompt's reply markdown, the only output editable as text"
    if m.msg_type != sprompt: raise ValueError(f"{m.id}: out=True edits a prompt's reply; assign `msg.output` directly for other types")
    return m.ai_res

def _set_out_text(m, s): m.output = s

In [ ]:
#| export
_msg_edit_doc = """
Be sure you've viewed the message (e.g. `view_msg`) so you know the line nums.

Message editing standard parameters:
- `id`: message id (or list of ids, or 'all'), unique prefixes allowed
- `dlg`: `Dialog` or ipynb path; the current dialog file if None
- `out`: edit the output text (prompt reply, or code outputs literal) instead of the source

returns: diff of changes, or "none: No changes.", or "error: ..."
"""

def _msg_edit(f, name=None):
    def wrapper(id, *args, dlg=None, out:bool=False, **kw):
        d, sv = _load_dlg(dlg)
        def _one(cid):
            m = d.msg(cid)
            try:
                text = _out_text(m) if out else m.content
                if not text: return f"error: Message has no {'output' if out else 'source'}"
                new = f(text, *args, **kw)
                if out: _set_out_text(m, new)
                else: m.content = new
            except Exception as e: return f'error: {e}'
            return str_diff(text, new) or 'none: No changes.'
        if isinstance(id, list) or id=='all':
            if id=='all': id = [m.id for m in d.messages]
            res = [(cid, r) for cid in id if not (r := _one(cid)).startswith('none:')]
        else: res = PrettyString(_one(id))
        _save(d, sv)
        return res
    res = splice_sig(wrapper, f, 'text')
    if name: res.__name__ = res.__qualname__ = name
    res.__doc__ = (f.__doc__ or '') + _msg_edit_doc
    return res

msg_insert_line   = _msg_edit(insert_line,   'msg_insert_line')
msg_str_replace   = _msg_edit(str_replace,   'msg_str_replace')
msg_strs_replace  = _msg_edit(strs_replace,  'msg_strs_replace')
msg_replace_lines = _msg_edit(replace_lines, 'msg_replace_lines')
msg_del_lines     = _msg_edit(del_lines,     'msg_del_lines')

In [ ]:
(tdir/'edit.ipynb').write_bytes((tdir/'demo.ipynb').read_bytes())  # scratch copy: demo.ipynb stays pristine for later sections
set_dlg(tdir/'edit.ipynb')
diff = msg_str_replace(calc.id, 'x = 21', 'x = 40')
assert '-x = 21' in str(diff) and '+x = 40' in str(diff)
test_eq(read_ipynb(tdir/'edit.ipynb').msg(calc.id).content, 'x = 40\nx*2')

In [ ]:
diff = msg_str_replace(ask.id, '**42**', '**80**', out=True)
assert '+It is **80**.' in str(diff)
test_eq(read_ipynb(tdir/'edit.ipynb').msg(ask.id).ai_res, 'It is **80**.')
assert str(msg_str_replace(calc.id, "'42'", "'80'", out=True)).startswith('error:')
assert str(msg_del_lines(setup.id, 1, 1, out=True)).startswith('error:')
bulk = msg_str_replace([setup.id, bad.id], '1/0', 'pass')
test_eq(len(bulk), 2)  # bulk mode reports errors alongside diffs
assert 'error' in bulk[0][1] and '+pass' in bulk[1][1]

In [ ]:
#| export
def python_msgs(
    func, # `str -> str` applied to each message's content
    *ids, # Messages or ids; all code messages if none given
    dlg=None, # A `Dialog`, or ipynb path; the current dialog file if None
):
    "Rewrite message sources with `func`, returning `(id, diff)` pairs for changed messages"
    d, sv = _load_dlg(dlg)
    ms = [d.msg(i) for i in ids] if ids else [m for m in d.messages if m.msg_type==scode]
    res = []
    for m in ms:
        new = func(m.content)
        if new != m.content:
            res.append((m.id, str_diff(m.content, new)))
            m.content = new
    _save(d, sv)
    return res

def ast_msgs(
    repls:list, # (pattern, replacement) ast-grep rules
    *ids, # Messages or ids; all code messages if none given
    dlg=None, # A `Dialog`, or ipynb path; the current dialog file if None
):
    "Apply ast-grep `repls` to message sources (requires the optional `remold` package)"
    return python_msgs(lambda t: ast_replace(t, repls), *ids, dlg=dlg)

In [ ]:
changed = python_msgs(lambda t: t.replace('x', 'y'), dlg=d)
test_eq(len(changed), 1)  # only `calc` contains an x
assert d.msg(calc.id).content.startswith('y = ')

## Structural search

Text search fights Python syntax: `cfg =` misses `cfg, aux = load()`, and a loose pattern on a two-letter name floods. The finder trio turns remold's structural queries into predicates for `find_msgs(pred=)`: `symdef_finder` matches messages that bind a name in their top-level scope, however Python spells the binding (tuple unpacking, `for`, `with ... as`, walrus, imports, `def`/`class`); `symref_finder` matches those that read it; and `ast_finder` matches an ast-grep pattern. A message that doesn't parse as Python never matches, so prose cells exclude themselves.

In [ ]:
#| export
def symdef_finder(name):
    "`find_msgs` predicate: does the message bind `name` at cell top level? (requires `remold`)"
    from remold import symdefs
    return lambda m: name in symdefs(m.content)

def symref_finder(name):
    "`find_msgs` predicate: does the message reference `name`? (requires `remold`)"
    from remold import symrefs
    return lambda m: name in symrefs(m.content)

def ast_finder(pattern):
    "`find_msgs` predicate: does ast-grep `pattern` match in the message? (requires `remold`)"
    from remold import astfind
    return lambda m: bool(astfind(m.content, pattern))

In [ ]:
fd = Dialog('finders')
f1 = fd.mk_message('cfg, aux = load_cfg()', msg_type=scode)
f2 = fd.mk_message('print(render(cfg))', msg_type=scode)
fd.mk_message('cfg is set up above', msg_type=snote)
test_eq(find_msgs(dlg=fd, pred=symdef_finder('cfg'), context=0), [f1])
test_eq(find_msgs(dlg=fd, pred=symref_finder('cfg'), context=0), [f2])
test_eq(find_msgs(dlg=fd, pred=ast_finder('render($_)'), context=0), [f2])

## Hash-verified edits

`lnhashview_msg` and `exhash_msg` bring exhash's stale-context-proof addressing to an in-memory message. `lnhashview_msg` is sugar for `view_msg(..., lnhashs=True)`; `exhash_msg` imports `exhash` lazily, so llmsurgery carries no hard dependency on it:

In [ ]:
#| export
def lnhashview_msg(
    m, # A `Message`, or an id looked up in `dlg`
    dlg=None, # `Dialog` or path when `m` is an id; the current dialog file if None
):
    "Hash-addressed view of `m.content`, for `exhash_msg`"
    return view_msg(m, dlg, lnhashs=True)

def exhash_msg(
    m, # A `Message`, or an id looked up in `dlg`
    *cmds:tuple, # exhash command tuples, addresses from `lnhashview_msg`
    sw:int=4, # Shift width for indent commands
    dlg=None, # `Dialog` or path when `m` is an id; the current dialog file if None
):
    "Apply exhash commands to `m.content`, returning the diff"
    from exhash import exhash
    d,sv = (None,False) if isinstance(m, Message) else _load_dlg(dlg)
    if d is not None: m = d.msg(m)
    res = exhash(m.content, list(cmds), sw=sw)
    m.content = '\n'.join(res.lines)
    _save(d, sv)
    return PrettyString(res.format_diff(context=1))

In [ ]:
diff = exhash_msg(calc, (lnhash(1, 'y = 21'), 's', '21', '12'))
test_eq(calc.content, 'y = 12\ny*2')
with expect_fail(Exception, 'stale'): exhash_msg(calc, (lnhash(1, 'wrong text'), 'd'))

Both also take a message id in place of a live `Message`, resolved through `dlg` like `view_msg` (the current dialog file when None). An id-based edit saves back to a file-backed dialog automatically:

In [ ]:
msg_replace_lines(calc.id, new_content='a = 1\na*2')
assert 'a = 1' in str(lnhashview_msg(calc.id))
diff = exhash_msg(calc.id, (lnhash(1, 'a = 1'), 's', '1', '9'))
test_eq(read_ipynb(tdir/'edit.ipynb').msg(calc.id).content, 'a = 9\na*2')

## Working with dialog files

Every function above takes `dlg` as a `Dialog`, a path, or None meaning the current dialog file (`set_dlg`). Operations persist automatically only when the dialog is file-backed, so views never dirty a file, edits on a loaded file save themselves, and edits on an in-memory `Dialog` are the caller's to save. `write_ipynb`'s byte-stability keeps the resulting diffs minimal. A few functions below exist only for the file world:

In [ ]:
set_dlg(tdir/'demo.ipynb')
assert '<prompt' in str(view_dlg())
assert str(summary_dlg()).count('\n')==3
s = str(summary_dlg(find_msgs(r'x\*2', msg_type=scode, context=0)))
assert s and s.count('\n')==0  # renders a find result too: exactly one line (empty would pass a bare count check)
f = view_dlg(find_msgs(r'x\*2', msg_type=scode, context=0), incl_out=True)
assert '<code' in str(f) and 'Setup' not in str(f)
xd.mk_message('', msg_type=sraw, meta=dict(rec_kind='system', rec={}))
assert 'kind="system"' in str(view_dlg(xd))  # tagged raws are badged, never shown as ordinary content

In [ ]:
#| export
def add_msg(
    source:str, # source for the new message
    msg_type:str='code', # 'code', 'note', 'prompt', or 'raw'
    before:str=None, # message or id to insert before
    after:str=None, # message or id to insert after
    dlg=None, # A `Dialog`, or ipynb path; the current dialog file if None
):
    "Add a new message before/after an existing one (pass exactly one), returning it"
    d, sv = _load_dlg(dlg)
    m = d.mk_message(source, msg_type=msg_type,
        before=None if before is None else d.msg(before), after=None if after is None else d.msg(after))
    _save(d, sv)
    return m

def del_msgs(
    *ids, # Messages or ids to delete
    dlg=None, # A `Dialog`, or ipynb path; the current dialog file if None
):
    "Delete messages by id, returning the removed messages"
    d, sv = _load_dlg(dlg)
    res = Msgs(d.remove_msgs([d.msg(i) for i in ids]))
    _save(d, sv)
    return res

def create_dlg(
    fname:str, # path for the new ipynb; must not already exist
    source:str='', # source for its first message
    msg_type:str='code', # 'code', 'note', 'prompt', or 'raw'
):
    "Create a new dialog file containing one message, returning the `Dialog` (with `path_` stamped)"
    f = Path(fname)
    if f.exists(): raise FileExistsError(str(f))
    d = Dialog(f.stem)
    d.mk_message(source, msg_type=msg_type)
    write_ipynb(d, f)
    d.path_ = f
    return d

In [ ]:
nm = add_msg('print(9)', after=calc.id)
test_eq(read_ipynb(tdir/'demo.ipynb').msg(nm).content, 'print(9)')
test_eq(del_msgs(nm)[0].content, 'print(9)')
with expect_fail(KeyError, 'no message'): read_ipynb(tdir/'demo.ipynb').msg(nm)
nd = create_dlg(tdir/'new.ipynb', '# hi', 'note')
test_eq(nd.messages[0].msg_type, snote)
test_eq(read_ipynb(tdir/'new.ipynb').msg(nd.messages[0]).content, '# hi')

In [ ]:
s1, s2 = split_msg(calc.id, 2)
test_eq(read_ipynb(tdir/'demo.ipynb').msg(s2).content, 'x*2')
merge_msgs(s1, s2)
test_eq(read_ipynb(tdir/'demo.ipynb').msg(s1).content, 'x = 21\n\nx*2')
copy_msgs(s1)
pids = paste_msgs(after=s1)
test_eq(read_ipynb(tdir/'demo.ipynb').msg(pids[0].id).content, 'x = 21\n\nx*2')
del_msgs(pids[0].id)
python_msgs(lambda t: t.replace('21', '40'), s1)
test_eq(read_ipynb(tdir/'demo.ipynb').msg(s1).content, 'x = 40\n\nx*2')

Under IPython, the `%%add_msg` magic takes the message body verbatim (no Python quoting): `%%add_msg [<path>] before=<id>|after=<id> [code|note|prompt|raw]`.

In [ ]:
#| export
def add_msg_magic(line, cell):
    "Add a new message with the magic body as its source, taken verbatim."
    mt, dlg, kw = 'code', None, {}
    for t in shlex.split(line):
        if '=' in t: kw.update([t.split('=', 1)])
        elif t in smsg_types: mt = t
        else: dlg = t
    return add_msg(cell.rstrip('\n'), msg_type=mt, dlg=dlg, **kw)

def load_ipython_extension(ipython): ipython.register_magic_function(add_msg_magic, 'cell', 'add_msg')

try:
    from IPython import get_ipython
    if (_ip := get_ipython()): load_ipython_extension(_ip)
except ImportError: pass